# Chilbolton Surface Meteorology Explorer

Runs the Dash app via JupyterLab's built-in proxy — no SSH port forwarding needed.

A clickable link will appear in the last cell's output. Interrupt the kernel to stop the server.


In [1]:
import os
import sys

PORT = 8050
HOST = "127.0.0.1"

# Suppress the GDAL/PROJ "Open of .../share/proj failed" warning that appears
# when geopandas initialises outside its conda activation context.
_conda_prefix = os.environ.get("CONDA_PREFIX", os.path.dirname(os.path.dirname(sys.executable)))
os.environ.setdefault("PROJ_DATA", os.path.join(_conda_prefix, "share", "proj"))


'/opt/envs/notebook-root/share/proj'

In [2]:
import os

# Compute the proxy subpath before importing app so that dash.Dash() picks it
# up from the environment variable at construction time.
base = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/")
proxy_url = f"{base}proxy/{PORT}/"
os.environ["DASH_REQUESTS_PATHNAME_PREFIX"] = proxy_url

# Import the Dash app object from app.py (does NOT start the server).
# Set any data-root overrides here before the import, e.g.:
#   os.environ['PRESSURE_DATA_ROOT'] = '/path/to/data'
from wx2026_chilbolton_dash import app


In [3]:
import signal
import socket
import threading
from IPython.display import display, HTML


# ── 1. Free the port if a previous instance is still running ─────────────────
def _free_port(port):
    port_hex = f"{port:04X}"
    inode = None
    for tcp_file in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            with open(tcp_file) as f:
                for line in f.readlines()[1:]:
                    cols = line.split()
                    if cols[1].split(":")[1] == port_hex and cols[3] == "0A":
                        inode = cols[9]
                        break
        except FileNotFoundError:
            pass
        if inode:
            break
    if not inode:
        return
    for pid in os.listdir("/proc"):
        if not pid.isdigit():
            continue
        try:
            for fd in os.listdir(f"/proc/{pid}/fd"):
                if os.readlink(f"/proc/{pid}/fd/{fd}") == f"socket:[{inode}]":
                    os.kill(int(pid), signal.SIGKILL)
                    print(f"Killed stale process {pid} on port {port}")
                    return
        except (PermissionError, FileNotFoundError, OSError):
            pass

_free_port(PORT)


# ── 2. Start server in a background thread, capture any startup errors ────────
_errors = []

def _run():
    try:
        app.run(host=HOST, port=PORT, debug=False, use_reloader=False)
    except Exception as e:
        _errors.append(e)

_thread = threading.Thread(target=_run, daemon=True)
_thread.start()


# ── 3. Wait until port is actually open (up to 10 s) ─────────────────────────
import time
for _ in range(20):
    if _errors:
        print(f"✗ Server failed to start: {_errors[0]}")
        break
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", PORT)) == 0:
            print(f"✓ Server listening on port {PORT}")
            break
    time.sleep(0.5)
else:
    print(f"✗ Server did not bind to port {PORT} within 10 s")

display(HTML(
    f'Open <a href="{proxy_url}" target="_blank">{proxy_url}</a> in your browser.<br>'
    f'<small>Restart the kernel to stop the server.</small>'
))


✓ Server listening on port 8050


In [4]:
import os
import subprocess
import sys

# Walk up the process tree from this kernel to find the Python running JupyterLab.
# The server process will have 'jupyter' in its command line.
def _find_server_python():
    pid = os.getpid()
    for _ in range(15):
        try:
            with open(f"/proc/{pid}/status") as f:
                ppid = int(next(l for l in f if l.startswith("PPid:")).split()[1])
            cmdline = open(f"/proc/{ppid}/cmdline").read().replace("\x00", " ")
            if "jupyter" in cmdline.lower():
                return os.readlink(f"/proc/{ppid}/exe"), ppid
            pid = ppid
        except OSError:
            break
    return None, None

server_python, server_pid = _find_server_python()

if server_python:
    print(f"JupyterLab server PID    : {server_pid}")
    print(f"JupyterLab server Python : {server_python}")
    print()

    # Check if jupyter-server-proxy is already there
    check = subprocess.run(
        [server_python, "-c", "import jupyter_server_proxy; print('already installed')"],
        capture_output=True, text=True,
    )
    if "already installed" in check.stdout:
        print("✓ jupyter-server-proxy is already installed in the server environment.")
        print("  If the proxy still returns 404, restart your JupyterLab session.")
    else:
        print("Installing jupyter-server-proxy into the server environment …")
        result = subprocess.run(
            [server_python, "-m", "pip", "install", "--quiet", "jupyter-server-proxy"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print("✓ Installed successfully.")
            print("  ⚠ You must now RESTART your JupyterLab session (File → Hub Control Panel")
            print("    → Stop My Server → Start Server) for the extension to be registered.")
        else:
            print(f"✗ Installation failed (permission denied or network issue).")
            print(result.stderr[-800:])
            print()
            print("  If you cannot install into the server environment, use SSH port")
            print("  forwarding instead:")
            print("    ssh -L 8050:localhost:8050 <your-jasmin-login-node>")
            print("  then open http://localhost:8050 in your local browser.")
else:
    print("Could not identify the JupyterLab server process.")
    print(f"Kernel Python: {sys.executable}")
    print()
    print("Manual option — run in a JASMIN terminal:")
    print("  pip install jupyter-server-proxy")
    print("then restart your JupyterLab session.")


JupyterLab server PID    : 7
JupyterLab server Python : /opt/envs/notebook-root/bin/python3.14

✓ jupyter-server-proxy is already installed in the server environment.
  If the proxy still returns 404, restart your JupyterLab session.
